# RLVR Verifier Walkthrough

This notebook explains the verifier step by step on two synthetic corner-case model outputs:
- one correct symbolic trace
- one incorrect but still parseable symbolic trace

It is designed for both local use and Google Colab, following the same clone-or-pull workflow as the training notebook.

In [ ]:
# Set RUN_ENV to 'local' on your computer or 'colab' in Google Colab.
RUN_ENV = 'colab'

GIT_REPO_URL = 'https://github.com/ffbskt/AgentAI.git'
LOCAL_REPO_DIR = r'D:/Codex projects/Transformer_Graph3/arithmetic-transformer'
COLAB_PROJECT_ROOT = '/content/drive/MyDrive/AgentAI'

In [ ]:
if RUN_ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

if RUN_ENV == 'colab':
    project_root = Path(COLAB_PROJECT_ROOT)
    repo_root = project_root
    git_dir = repo_root / '.git'

    if git_dir.exists():
        print(f"'{repo_root}' is already a git repository. Pulling latest changes...")
        subprocess.run(['git', '-C', str(repo_root), 'pull'], check=True)
    elif repo_root.exists():
        print(f"'{repo_root}' exists but is not a git repository. Removing and re-cloning...")
        shutil.rmtree(repo_root)
        project_root.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', GIT_REPO_URL, str(repo_root)], check=True)
    else:
        print(f"Cloning '{GIT_REPO_URL}' into '{repo_root}'...")
        project_root.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', GIT_REPO_URL, str(repo_root)], check=True)

    REPO_DIR = repo_root / 'arithmetic-transformer'
else:
    REPO_DIR = Path(LOCAL_REPO_DIR)

print('Repo dir:', REPO_DIR)

In [ ]:
%cd {REPO_DIR}
!python -m pip install -q -r requirements-colab.txt

## Load synthetic corner cases and verifier functions

These synthetic outputs simulate model generations on a carry-chain corner case:
- problem: `99+1`
- correct trace: `99+1=90+9+1=90+10=100`
- incorrect trace: `99+1=90+9+1=90+11=101`

In [ ]:
from rlvr_tiny.tests.synthetic_outputs import (
    CORNER_CASE_CORRECT_TRACE,
    CORNER_CASE_INCORRECT_TRACE,
    CORNER_CASE_PROBLEM,
)
from rlvr_tiny.verify import (
    RewardConfig,
    check_local_steps,
    eval_expr,
    extract_final_answer,
    parse_trace,
    score_trace,
)

print('Problem:', CORNER_CASE_PROBLEM)
print('Correct trace:', CORNER_CASE_CORRECT_TRACE)
print('Incorrect trace:', CORNER_CASE_INCORRECT_TRACE)

## Why `parse_trace` matters for RLVR

RLVR reward should not be given to outputs that are malformed.
`parse_trace` is the first gate:
- split the symbolic trajectory into `=`-separated parts
- detect empty segments or illegal characters
- produce a simple parse status and error code

If parsing fails, the RL reward should drop immediately.

In [ ]:
for name, trace in [('correct', CORNER_CASE_CORRECT_TRACE), ('incorrect', CORNER_CASE_INCORRECT_TRACE)]:
    parts, ok, err = parse_trace(trace)
    print('\nTRACE TYPE:', name)
    print('trace:', trace)
    print('parts:', parts)
    print('parse_ok:', ok)
    print('error:', err)

## Why `eval_expr` matters for RLVR

`eval_expr` turns symbolic arithmetic expressions into deterministic values.
This is how the verifier checks whether intermediate rewrites preserve meaning.

Without this function, the reward could not distinguish valid symbolic reasoning from arbitrary string edits.

In [ ]:
for name, trace in [('correct', CORNER_CASE_CORRECT_TRACE), ('incorrect', CORNER_CASE_INCORRECT_TRACE)]:
    parts, _, _ = parse_trace(trace)
    print('\nTRACE TYPE:', name)
    for i, part in enumerate(parts):
        print(f'part[{i}] = {part!r} -> value = {eval_expr(part)}')

## Why `check_local_steps` matters for RLVR

Final-answer reward alone is not enough.
A model could output a correct final answer with invalid intermediate steps.

`check_local_steps` gives process supervision:
- compare each adjacent pair of expressions
- mark each step valid or invalid
- compute the valid-step fraction

This is the core process-level signal used to reward real reasoning traces.

In [ ]:
for name, trace in [('correct', CORNER_CASE_CORRECT_TRACE), ('incorrect', CORNER_CASE_INCORRECT_TRACE)]:
    parts, _, _ = parse_trace(trace)
    flags, fraction = check_local_steps(parts, fmt='B')
    print('\nTRACE TYPE:', name)
    for i, (left, right) in enumerate(zip(parts[:-1], parts[1:])):
        print(f'step[{i}]: {left}  ->  {right}   valid={flags[i]}')
    print('valid_step_fraction:', fraction)

## Why `extract_final_answer` matters for RLVR

The project rule says the final answer is always the substring after the last `=`.
`extract_final_answer` enforces that convention so training, evaluation, and reward all agree on the same target.

In [ ]:
for name, trace in [('correct', CORNER_CASE_CORRECT_TRACE), ('incorrect', CORNER_CASE_INCORRECT_TRACE)]:
    parts, _, _ = parse_trace(trace)
    print('\nTRACE TYPE:', name)
    print('final answer:', extract_final_answer(parts))

## Why `score_trace` matters for RLVR

`score_trace` combines all verifier pieces into one structured reward record.
It returns:
- syntax status
- final-answer correctness
- number of steps
- local-step-valid fraction
- total reward
- error type

This is the function the RLVR loop can call directly when scoring sampled trajectories.

In [ ]:
reward_cfg = RewardConfig()

for name, trace in [('correct', CORNER_CASE_CORRECT_TRACE), ('incorrect', CORNER_CASE_INCORRECT_TRACE)]:
    result = score_trace(CORNER_CASE_PROBLEM, trace, reward_cfg=reward_cfg, fmt='B')
    print('\nTRACE TYPE:', name)
    for key, value in result.items():
        print(f'{key}: {value}')

## Interpretation

Expected behavior:
- the correct trace should be parseable, step-valid, final-correct, and receive higher reward
- the incorrect trace should still parse, but should lose reward because one middle step is invalid and the final answer is wrong

This is exactly why each function is needed for RLVR:
- `parse_trace`: reject malformed trajectories
- `eval_expr`: deterministic arithmetic semantics
- `check_local_steps`: process reward
- `extract_final_answer`: consistent final target extraction
- `score_trace`: one structured interface for training and evaluation